In [2]:
!pip install fair-esm

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import torch
import esm
import pandas as pd
import numpy as np
from tqdm import tqdm

In [4]:
# ===============================
# CONFIG
# ===============================

MODEL_NAME = "esm2_t33_650M_UR50D"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [5]:
# ===============================
# LOAD MODEL
# ===============================

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

model.eval()
model = model.to(DEVICE)

LAYER = 33  # last layer

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to C:\Users\Nitro/.cache\torch\hub\checkpoints\esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to C:\Users\Nitro/.cache\torch\hub\checkpoints\esm2_t33_650M_UR50D-contact-regression.pt


In [6]:
# ===============================
# LOAD DATA
# ===============================

# archivo con columna 'sequence'
df = pd.read_csv("consolidated-PD-L1-1772591316870.csv")

# ejemplo de secuencia:
# GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTNEIFYCTFRRLDPEENHTAELVIPELP/EVTLTGSLEEPLLP


# ===============================
# FUNCTION: GENERATE EMBEDDINGS
# ===============================

def generate_embeddings(sequences):

    all_embeddings = []

    for seq in sequences:

        # separar cadenas
        chains = seq.split("/")

        chain_embeddings = []

        for chain in chains:

            data = [("protein", chain)]

            labels, strs, tokens = batch_converter(data)
            tokens = tokens.to(DEVICE)

            with torch.no_grad():
                outputs = model(tokens, repr_layers=[LAYER])

            reps = outputs["representations"][LAYER]

            seq_len = len(chain)

            # quitar BOS token
            emb = reps[0, 1:seq_len+1]

            # mean pooling
            emb = emb.mean(0)

            chain_embeddings.append(emb.cpu().numpy())

        # concatenar cadenas
        final_embedding = np.concatenate(chain_embeddings)

        all_embeddings.append(final_embedding)

    return np.array(all_embeddings)



In [8]:

# ===============================
# GENERATE
# ===============================

embeddings = []

for i in tqdm(range(0, len(df), BATCH_SIZE)):

    batch_seqs = df["seq"].iloc[i:i+BATCH_SIZE].tolist()

    batch_emb = generate_embeddings(batch_seqs)

    embeddings.append(batch_emb)

embeddings = np.vstack(embeddings)


# ===============================
# SAVE
# ===============================

np.save("protein_embeddings.npy", embeddings)

print("Embeddings shape:", embeddings.shape)

100%|██████████| 900/900 [1:49:09<00:00,  7.28s/it]


Embeddings shape: (28800, 2560)
